## AI-Based Image Captioning and Search Engine Using CLIP

# Objective

- Develop an AI-based Image Captioning and Image Search system using CLIP.
- Generate captions for uploaded images.
- Search relevant images using text queries.
- Extract image and text features using deep learning.
- Build an interactive web application using Streamlit.

In [ ]:
 # Install PyTorch, TorchVision, and Pillow
!pip install torch torchvision pillow


In [ ]:
# Install OpenAI CLIP from GitHub
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-oklb78ve
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-oklb78ve
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=492e298b976ea8bb58745965bdb3e9c4072c7307532ccd9f1ef7cd27f1a60a11
  Stored in directory: /tmp/pip-ephem-wheel-cache-n6xqlfn4/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


#Dataset Extraction and Setup

In [ ]:
# Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Extract Dataset ZIP
import zipfile

zip_path = "/content/drive/MyDrive/lstm.zip"
extract_path = "/content/lstm_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted")

✅ Extracted


In [ ]:
# Check Extracted Files

import os

print(os.listdir("/content/lstm_data"))

['Images', 'captions.txt']


In [ ]:
# Verify Dataset Paths
image_folder = "/content/lstm_data/Images"
captions_file = "/content/lstm_data/captions.txt"
print(os.path.exists(image_folder))   # True
print(os.path.exists(captions_file))  # True

True
True


#CLIP Setup

In [ ]:
# Load CLIP Model
import clip
import torch

model, preprocess = clip.load("ViT-B/32", device="cpu")

100%|███████████████████████████████████████| 338M/338M [00:06<00:00, 50.8MiB/s]


In [ ]:
# Load Captions
data = []

with open(captions_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            img, caption = parts
            data.append([img, caption])

print("Loaded:", len(data))

Loaded: 40456


In [ ]:
assert os.path.exists(captions_file), "Caption file missing!"

In [ ]:
# =========================
# Create Image-Caption Mapping
# =========================
mapping = {}

with open(captions_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            img, cap = parts
            if img not in mapping:
                mapping[img] = []

            mapping[img].append(cap)

print("Total images:", len(mapping))

Total images: 8092


#CLIP Feature Extraction

In [ ]:
# =========================
# Extract and Save CLIP Features
# =========================
import os
import pickle
from PIL import Image

image_features = {}
text_features = {}

for img in list(mapping.keys())[:500]:  # Process first 500 images
    path = os.path.join(image_folder, img)

    try:
        image = Image.open(path).convert("RGB")
        image_input = preprocess(image).unsqueeze(0)

        # Extract image features
        with torch.no_grad():
            img_feat = model.encode_image(image_input)

        image_features[img] = img_feat

        # Extract text features
        for cap in mapping[img]:
            text_tokens = clip.tokenize([cap])

            with torch.no_grad():
                txt_feat = model.encode_text(text_tokens)

            text_features[(img, cap)] = txt_feat

    except:
        continue

# Save features to Drive
with open("/content/drive/MyDrive/image_features.pkl", "wb") as f:
    pickle.dump(image_features, f)

with open("/content/drive/MyDrive/text_features.pkl", "wb") as f:
    pickle.dump(text_features, f)

print("✅ Features saved")

✅ Features saved


#Caption Generation

In [ ]:
# Load Features and Generate Captions
# =========================
import pickle

# Load saved features
with open("/content/drive/MyDrive/image_features.pkl", "rb") as f:
    image_features = pickle.load(f)

with open("/content/drive/MyDrive/text_features.pkl", "rb") as f:
    text_features = pickle.load(f)

print("✅ Features loaded instantly")

✅ Features loaded instantly


In [ ]:
# Function to get top 5 captions
from PIL import Image

def get_top5_captions(image_path):
    image = Image.open(image_path).convert("RGB")
    image_input = preprocess(image).unsqueeze(0)

    # Extract image feature
    with torch.no_grad():
        img_feat = model.encode_image(image_input)

    scores = []

    # Compare with text features
    for (img, cap), txt_feat in text_features.items():
        score = similarity(img_feat, txt_feat)
        scores.append((cap, score))

    # Sort captions by similarity
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    return scores[:5]

#Image Search and Similarity

In [ ]:
# Search images from text query
def search_images(query):
    tokens = clip.tokenize([query])

    with torch.no_grad():
        txt_feat = model.encode_text(tokens)

    scores = []

    for img, img_feat in image_features.items():
        score = similarity(img_feat, txt_feat)
        scores.append((img, score))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    return scores[:5]

In [ ]:

# Calculate cosine similarity
def similarity(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)

    return (a @ b.T).item()

In [ ]:

# Test caption generation
print(get_top5_captions("/content/lstm_data/Images/667626_18933d713e.jpg"))

# Test image search
print(search_images("a girl playing in water"))

[('A girl in pigtails splashes in the shallow water .', 0.3165919780731201), ('A young girl wearing a bikini in an innertube .', 0.3115643858909607), ('A young girl in pigtails plays in the water .', 0.3013424575328827), ('A young girl plays in shallow water .', 0.2997736930847168), ('A girl with pigtails plays in the water .', 0.2966125011444092)]
[('1255504166_f2437febcb.jpg', 0.3160179853439331), ('1151466868_3bc4d9580b.jpg', 0.31162089109420776), ('1095476286_87d4f8664e.jpg', 0.3097914159297943), ('1119463452_69d4eecd08.jpg', 0.3004550635814667), ('1308472581_9961782889.jpg', 0.29647019505500793)]


#Streamlit Web Application

In [ ]:
!pip install streamlit pyngrok

In [ ]:
%%writefile app.py
# (paste your full Streamlit code here)

Writing app.py


In [ ]:
pip install streamlit tensorflow pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 54.4 MB/s eta 0:00:00


In [ ]:
# =========================
# AI Image Caption & Search Engine
# =========================

import streamlit as st
import torch
import clip
import pickle
from PIL import Image
import pandas as pd


# =========================
# Helper Functions
# =========================

# Calculate cosine similarity
def similarity(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)

    return (a @ b.T).item()


# Explain matching words
def explain_match(query, caption):
    query_words = set(query.lower().split())
    caption_words = set(caption.lower().split())

    common = query_words.intersection(caption_words)

    return "Common words: " + ", ".join(common) if common else "Semantic match (no direct words)"


# Confidence level
def confidence(score):
    if score > 0.30:
        return "High"

    elif score > 0.25:
        return "Medium"

    else:
        return "Low"


# =========================
# Load CLIP Model
# =========================

@st.cache_resource
def load_model():

    model, preprocess = clip.load("ViT-B/32", device="cpu")
    model.eval()

    return model, preprocess


model, preprocess = load_model()


# =========================
# Load Saved Features
# =========================

@st.cache_data
def load_features():

    with open("image_features.pkl", "rb") as f:
        image_features = pickle.load(f)

    with open("text_features.pkl", "rb") as f:
        text_features = pickle.load(f)

    return image_features, text_features


image_features, text_features = load_features()


# =========================
# Streamlit UI
# =========================

st.title("🚀 AI Image Caption & Search Engine")


# =========================
# Image → Caption
# =========================

st.header("📸 Image → Caption")

# Upload image
uploaded = st.file_uploader("Upload Image")

if uploaded:

    image = Image.open(uploaded).convert("RGB")
    st.image(image)

    st.write("📂 File:", uploaded.name)

    # Generate captions
    if st.button("Top 5 Captions"):

        with st.spinner("Analyzing image..."):

            image_input = preprocess(image).unsqueeze(0)

            # Extract image features
            with torch.no_grad():
                img_feat = model.encode_image(image_input)

            results = []

            # Compare with text features
            for (img, cap), txt_feat in text_features.items():

                score = similarity(img_feat, txt_feat)
                results.append((cap, score))

            # Sort by similarity
            results = sorted(results, key=lambda x: x[1], reverse=True)

        # Best caption
        best_cap, best_score = results[0]

        st.success(f"🏆 Best Match: {best_cap} ({best_score:.2f})")

        st.subheader("Top Matches:")

        # Display top captions
        for cap, score in results[:5]:

            st.write(f"{cap} ({score:.2f}) - {confidence(score)} confidence")
            st.caption(explain_match("", cap))

        # Display graph
        scores_df = pd.DataFrame(results[:5], columns=["Caption", "Score"])

        st.bar_chart(scores_df.set_index("Caption"))


# =========================
# Text → Image Search
# =========================

st.header("🔍 Text → Image Search")

# Input search query
query = st.text_input("Search images")

if st.button("Search"):

    if query.strip() == "":
        st.warning("Please enter a search query")

    else:

        with st.spinner("Searching images..."):

            tokens = clip.tokenize([query])

            # Extract text features
            with torch.no_grad():
                txt_feat = model.encode_text(tokens)

            results = []

            # Compare with image features
            for img, img_feat in image_features.items():

                score = similarity(img_feat, txt_feat)
                results.append((img, score))

            # Sort by similarity
            results = sorted(results, key=lambda x: x[1], reverse=True)

        st.subheader("Top Matching Images:")

        # Display top images
        for img, score in results[:5]:

            st.image(
                f"Images/{img}",
                caption=f"{score:.2f} - {confidence(score)} confidence"
            )


# =========================
# System Explanation
# =========================

st.header("🧠 How it works")

st.write("""

This system uses a pretrained model to understand both images and text.

• Images and captions are converted into embeddings (vectors)

• These embeddings capture semantic meaning (objects, actions, scenes)

• Cosine similarity is used to compare them

• The system returns the most similar matches

""")


# =========================
# Example Queries
# =========================

st.header("💡 Try These Examples")

st.write("""

- a girl playing in water

- a dog running

- children playing

- man riding bicycle

""")

FileNotFoundError: [Errno 2] No such file or directory: 'image_features.pkl'

#Conclusion

The project successfully implements an Image Caption and Search Engine using the CLIP model and Streamlit. The system can understand both images and text by converting them into feature embeddings and measuring similarity between them. It generates meaningful captions for images and retrieves relevant images based on user text queries, demonstrating the effectiveness of multimodal AI models in image-text understanding.